In [1]:
from datasets import load_dataset
import pandas as pd


import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from tqdm import tqdm
from gpt_arcitecture import GPTModel, load_params, download_and_load_gpt2, generate
import torch
import tiktoken
from torch.utils.data import DataLoader, random_split
from torch.nn.functional import cross_entropy

import warnings
warnings.filterwarnings("ignore")

torch.set_default_device("cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = generator = torch.Generator(device = device)

tokenizer = tiktoken.get_encoding("gpt2")

settings, vocab, params = download_and_load_gpt2("124M", "./archive")
model = GPTModel().to(device)
load_params(model,params)

eos_id = 50246
ignore_id = -100

2025-11-08 20:31:57.219989: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-08 20:31:57.506647: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-08 20:31:58.968028: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-08 20:31:58.968627: I external/local_xla/xla/tsl/cuda/cudart

File already exists and is up-to-date: ./archive/124M/checkpoint
File already exists and is up-to-date: ./archive/124M/encoder.json
File already exists and is up-to-date: ./archive/124M/hparams.json
File already exists and is up-to-date: ./archive/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: ./archive/124M/model.ckpt.index
File already exists and is up-to-date: ./archive/124M/model.ckpt.meta
File already exists and is up-to-date: ./archive/124M/vocab.bpe


In [2]:
train_df = load_dataset("databricks/databricks-dolly-15k")
train_df = train_df["train"].to_pandas()
print(train_df.columns)
print(train_df["category"].value_counts())

train_df = train_df.sample(frac=1).reset_index()

# prefix = "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n"
prefix = ""
train_df["formatted"] = prefix + "### Instruction:\n" + train_df["instruction"] + "\n### Context:\n" + train_df["context"]  + "\n### Response:\n" + train_df["response"]
train_encoded = list(map(tokenizer.encode, train_df["formatted"]))

Index(['instruction', 'context', 'response', 'category'], dtype='object')
category
open_qa                   3742
general_qa                2191
classification            2136
closed_qa                 1773
brainstorming             1766
information_extraction    1506
summarization             1188
creative_writing           709
Name: count, dtype: int64


In [3]:
def custom_collate(batch, allowed_max_len=settings["n_ctx"]):
    B = len(batch)

    lens = [min(len(x) + 1, allowed_max_len + 1) for x in batch]
    L = max(lens)

    # Preallocate once. Fill with sensible defaults.
    inputs  = torch.full((B, L - 1), eos_id,     dtype=torch.long)   # pad inputs with EOS
    targets = torch.full((B, L - 1), ignore_id,  dtype=torch.long)   # pad targets with IGNORE

    for i, seq in enumerate(batch):
        s = seq + [eos_id]
        s = s[:L]

        n = lens[i] - 1                # number of valid positions in inputs/targets
        if n <= 0:
            continue

        inputs[i, :n]  = torch.as_tensor(s[:-1][:n], dtype=torch.long)
        targets[i, :n] = torch.as_tensor(s[1:][:n],  dtype=torch.long)

    return inputs, targets

train_split, test_split, val_split = random_split(train_encoded, lengths = [0.85, 0.10,0.05])
BATCH_SIZE = 10
train_loader = DataLoader(train_split, batch_size = BATCH_SIZE, shuffle = True, drop_last=True , collate_fn=custom_collate)
test_loader = DataLoader(test_split, batch_size = BATCH_SIZE, shuffle = False , collate_fn=custom_collate)
val_loader = DataLoader(val_split, batch_size = BATCH_SIZE, shuffle = False , collate_fn=custom_collate)

In [ ]:
epochs = 10
history = []

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.1)

for epoch in range(10):
    torch.cuda.empty_cache()

    total_loss = 0
    for input_batch, target_batch in tqdm(train_loader):
        input_batch = input_batch.to(device)
        target_batch = target_batch.to(device)

        optimizer.zero_grad()

        pred_batch = model(input_batch).to(device)
        loss = cross_entropy(pred_batch.flatten(0,1), target_batch.flatten())
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    with torch.no_grad():
        avg_train_loss = total_loss/(len(train_loader)*BATCH_SIZE)

        total_loss = 0
        for input_batch, target_batch in val_loader:
            pred_batch = model(input_batch)
            loss = cross_entropy(pred_batch.flatten(0,1), target_batch.flatten())
            total_loss += loss.item()

        avg_val_loss = total_loss / (len(val_loader)*BATCH_SIZE)

        this_epoch_history = {
            "Epoch" : epochs+1,
            "Training Loss": avg_train_loss,
            "Validation Loss": avg_val_loss,
        }

        history.append(this_epoch_history)
        print(this_epoch_history)

  0%|          | 0/1276 [00:00<?, ?it/s]